# Stage A: Sensor A Data Preparation

**Update:** this stage originally reconstructed approximate per-reading
seconds from a minute-resolution export. The user has since obtained the
full device log (`Sensor_A_combined_file.xlsx`) with true second
resolution for every row, making interpolation unnecessary. This notebook
now just dedups Sensor A the same way Sensor B already was in Stage B.
Rerunning dedup on the corrected data changes the row count substantially
(the old minute-resolution dedup undercounted -- see design doc), so
Stages B and C are rebuilt after this one.

Design doc: `docs/specs/2026-07-29-stage-a-time-reconstruction-design.md`


In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../DATA"


## 1. Load & inspect raw data

In [2]:
df = pd.read_excel(f"{DATA_DIR}/Sensor_A_combined_file.xlsx")

print(f"raw rows: {len(df)}")
print(f"date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df["source_file"].value_counts()


raw rows: 945642
date range: 2026-02-26 11:51:41 to 2026-05-31 10:55:46


source_file
data_A_260526.csv          491377
data230526a.csv            306652
data_090526A.csv            52526
data_A300526.csv            43031
dataA_110426_1_good.csv     34125
data_A_020526.csv           15567
data_310526A.csv             2364
Name: count, dtype: int64

In [3]:
is_sorted = df["timestamp"].is_monotonic_increasing
print(f"timestamp column is globally non-decreasing: {is_sorted}")


timestamp column is globally non-decreasing: True


## 2. Deduplicate

Same duplicate-export issue as Sensor B (handled inline in Stage B):
several source files are cumulative re-exports of the same underlying
on-device log. Deduplicate globally on `(MAC, RSSI, timestamp)`.


In [4]:
pre_counts = df["source_file"].value_counts()

dedup_key = ["MAC", "RSSI", "timestamp"]
df_dedup = df.drop_duplicates(subset=dedup_key, keep="first").reset_index(drop=True)

post_counts = df_dedup["source_file"].value_counts()

report = pd.DataFrame({"pre_dedup": pre_counts, "post_dedup": post_counts}).fillna(0).astype(int)
report["removed"] = report["pre_dedup"] - report["post_dedup"]
report


,pre_dedup,post_dedup,removed
source_file,,,
data_A_260526.csv,491377,338054,153323
data230526a.csv,306652,152783,153869
data_090526A.csv,52526,52462,64
data_A300526.csv,43031,42972,59
dataA_110426_1_good.csv,34125,34077,48
data_A_020526.csv,15567,15542,25
data_310526A.csv,2364,2364,0


In [5]:
print(f"rows before dedup: {len(df)}")
print(f"rows after dedup:  {len(df_dedup)}")
print(f"duplicates removed: {len(df) - len(df_dedup)}")


rows before dedup: 945642
rows after dedup:  638254
duplicates removed: 307388


## 3. Validation / sanity checks

In [6]:
monotonic_check = df_dedup.groupby("source_file")["timestamp"].apply(lambda s: s.is_monotonic_increasing)
print("timestamp non-decreasing per source file:")
print(monotonic_check)
assert monotonic_check.all(), "found a source file where timestamps go backwards"


timestamp non-decreasing per source file:
source_file
data230526a.csv            True
dataA_110426_1_good.csv    True
data_090526A.csv           True
data_310526A.csv           True
data_A300526.csv           True
data_A_020526.csv          True
data_A_260526.csv          True
Name: timestamp, dtype: bool


In [7]:
# How many exact-duplicate-timestamp readings remain per MAC (two different
# devices/detections at the same second are expected and fine; this is just
# an FYI stat, not a validation failure).
same_second = df_dedup.groupby(["MAC", "timestamp"]).size()
print(f"reading groups sharing an identical (MAC, timestamp): {(same_second > 1).sum()} "
      f"out of {len(same_second)} groups")


reading groups sharing an identical (MAC, timestamp): 37596 out of 600651 groups


## 4. Save outputs

In [8]:
out_cols = ["MAC", "RSSI", "timestamp", "name", "manufacturer", "serviceUUID", "source_file"]
out = df_dedup[out_cols].copy()
# name/manufacturer/serviceUUID all have mixed types (some all-digit BLE
# values, e.g. "102000000", got read as int rather than str) -- normalize
# to string so parquet can write them.
for col in ["name", "manufacturer", "serviceUUID"]:
    out[col] = out[col].astype("string")

out.to_csv(f"{DATA_DIR}/sensor_A_deduped.csv", index=False)
out.to_parquet(f"{DATA_DIR}/sensor_A_deduped.parquet", index=False)

print(f"wrote {len(out)} rows to sensor_A_deduped.csv and .parquet")


wrote 638254 rows to sensor_A_deduped.csv and .parquet
